# YOLOv8 training notebook

Train a YOLOv8 detector used to find plant/crop regions for the downstream emergence pipeline.

All paths are built from `ROOT_PATH`, currently set to `/home/vasakjakub/sprout-emergence-prediction`.

## Inputs

- YOLO dataset with `train`, `val`, and `test` splits.
- `data.yaml` describing the dataset classes and image paths.
- Starting YOLOv8 weights, for example `yolov8n.pt`.

## Outputs

- Trained detector weights saved as `germination_detector.pt`.
- Training plots and validation artifacts under the YOLO run directory.
- Test-set predictions for a quick visual check.

## Basic workflow

1. Configure project paths and imports.
2. Preview one annotation from the training split.
3. Train the YOLOv8 model.
4. Rename the best checkpoint.
5. Evaluate and visualize predictions on the test split.

## Notes

Use a GPU environment when possible. Before running training, confirm that `data.yaml` points to the correct image and label folders.

# FUNCTIONS

### Paths and imports

This cell imports the required libraries and defines all project paths from `ROOT_PATH`.

In [ ]:
import random
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import ultralytics
from IPython.display import clear_output
from ultralytics import YOLO

# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("/home/vasakjakub/sprout-emergence-prediction")
OBJECT_DETECTION_MODEL_DIR = ROOT_PATH / "models" / "object_detection_model"
YOLO_FINE_TUNING_DIR = OBJECT_DETECTION_MODEL_DIR / "fine_tuning"
YOLO_ANNOTATIONS_DIR = OBJECT_DETECTION_MODEL_DIR / "detect_annotations"
DATA_YAML_PATH = YOLO_ANNOTATIONS_DIR / "data.yaml"

clear_output()

print("Repo root:", ROOT_PATH)
print("YOLO annotations:", YOLO_ANNOTATIONS_DIR)
print("Fine-tuning dir:", YOLO_FINE_TUNING_DIR)
print("data.yaml:", DATA_YAML_PATH)
ultralytics.checks()

if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f"Missing data.yaml: {DATA_YAML_PATH}")

In [ ]:
root = str(ROOT_PATH) + "/"
save_model = f"{YOLO_FINE_TUNING_DIR}/"
yolo_annotations = f"{YOLO_ANNOTATIONS_DIR}/"

### Hyperparameters

In [ ]:
n_epochs = 100

### Dataset structure

The annotation folder should follow the standard YOLO layout:

- `train/images` and `train/labels`
- `val/images` and `val/labels`
- `test/images` and `test/labels`
- `data.yaml` in the annotation root

### Training

Set `mod_type` to choose the YOLOv8 model size, for example `n`, `s`, or `m`. The notebook expects the matching weights file in the model directory.

In [ ]:
def train_model(
    yolo_annotations: str, save_model: str, n_epochs: int, mod_type: str
) -> None:
    """Train a YOLO model using the Ultralytics Python API.

    Args:
        yolo_annotations: Directory containing YOLO annotations and ``data.yaml``.
        save_model: Directory containing model weights and training outputs.
        n_epochs: Number of epochs used for YOLO training.
        mod_type: YOLOv8 model size suffix, for example ``n`` or ``s``.

    Author:
        Jakub Vašák
    """
    results_dir = Path(save_model) / "runs" / "detect"
    data_yaml = Path(yolo_annotations) / "data.yaml"

    if results_dir.exists():
        shutil.rmtree(results_dir)

    model_weights = Path(save_model) / f"yolov8{mod_type}.pt"
    model = YOLO(str(model_weights))
    model.train(
        data=str(data_yaml),
        epochs=n_epochs,
        imgsz=800,
        plots=True,
        project=str(results_dir),
        name="train_results",
        save_dir=save_model,
    )


def predict_images(path_to_pretrained_model: str, yolo_annotations: str) -> None:
    """Run inference with a pretrained YOLO model on the test images.

    Args:
        path_to_pretrained_model: Path to the saved YOLO model weights file.
        yolo_annotations: Directory containing the YOLO dataset split folders.

    Author:
        Jakub Vašák
    """
    test_images = Path(yolo_annotations) / "test" / "images"
    model = YOLO(path_to_pretrained_model)
    model.predict(
        str(test_images),
        save=True,
        conf=0.5,
        project=path_to_pretrained_model,
        name="predictions",
    )

In [ ]:
def visualize_random_annotation(yolo_annotations: str) -> None:
    """Visualize a random training image with its YOLO annotations.

    Args:
        yolo_annotations: Directory containing YOLO ``train/images`` and
            ``train/labels`` folders.

    Raises:
        FileNotFoundError: If OpenCV cannot load the selected random image.
    """
    images_dir = Path(yolo_annotations) / "train" / "images"
    labels_dir = Path(yolo_annotations) / "train" / "labels"

    image_files = [path for path in images_dir.iterdir() if path.is_file()]

    if not image_files:
        print("No images found in the training directory.")
        return

    random_image = random.choice(image_files)  # noqa: S311
    img = cv2.imread(str(random_image))
    if img is None:
        raise FileNotFoundError(f"Could not load image: {random_image}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img.shape

    label_path = labels_dir / f"{random_image.stem}.txt"

    print(f"Image: {random_image.name}")
    print(f"Dimensions: {width}x{height}")

    if label_path.exists():
        with label_path.open(encoding="utf-8") as handle:
            annotations = handle.readlines()

        print(f"Found {len(annotations)} annotations:")

        for annotation in annotations:
            values = annotation.strip().split()
            if len(values) >= 5:
                class_id = int(values[0])
                x_center = float(values[1]) * width
                y_center = float(values[2]) * height
                box_width = float(values[3]) * width
                box_height = float(values[4]) * height

                x1 = int(x_center - (box_width / 2))
                y1 = int(y_center - (box_height / 2))
                x2 = int(x_center + (box_width / 2))
                y2 = int(y_center + (box_height / 2))

                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    img,
                    f"Class {class_id}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2,
                )

                print(
                    f"  Class {class_id}: center=({x_center:.1f}, {y_center:.1f}), "
                    f"width={box_width:.1f}, height={box_height:.1f}"
                )
    else:
        print(f"No label file found for {random_image.name}")

    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

# Execution

Run the cells below in order. Training can take a while depending on the GPU and dataset size.

In [ ]:
# Check the saved annotations
visualize_random_annotation(yolo_annotations)

### Train YOLOv8n

This section trains the selected YOLOv8 model and saves the best checkpoint.

In [ ]:
# Set the type of the model
mod_type = "n"

train_model(yolo_annotations, save_model, n_epochs, mod_type)

In [ ]:
# Rename the best.pt file to the desired name
renamed_model = Path(save_model) / "germination_detector.pt"
best_model = (
    Path(save_model) / "runs" / "detect" / "train_results" / "weights" / "best.pt"
)
best_model.rename(renamed_model)

predict_images(str(renamed_model), yolo_annotations)

## Test set

Evaluate the saved detector on the YOLO test split and display a few predicted examples.

In [ ]:
def evaluate_test_images(
    path_to_pretrained_model: str, test_annotations_path: str, sample_count: int = 4
) -> None:
    """Run YOLO evaluation on the test split and plot predictions.

    Args:
        path_to_pretrained_model: Path to the saved YOLO model weights file.
        test_annotations_path: Path to the YOLO test split directory containing
            the ``images`` folder.
        sample_count: Maximum number of predicted test images to display.

    Raises:
        FileNotFoundError: If the test image directory, ``data.yaml``, or test
            images are missing.

    Author:
        Jakub Vašák
    """
    test_root = Path(test_annotations_path)
    test_images_dir = test_root / "images"
    data_yaml_path = test_root.parent / "data.yaml"

    if not test_images_dir.exists():
        raise FileNotFoundError(f"Missing test images directory: {test_images_dir}")

    if not data_yaml_path.exists():
        raise FileNotFoundError(f"Missing data.yaml file: {data_yaml_path}")

    image_paths = sorted(path for path in test_images_dir.iterdir() if path.is_file())
    if not image_paths:
        raise FileNotFoundError(f"No test images found in {test_images_dir}")

    model = YOLO(path_to_pretrained_model)

    metrics = model.val(
        data=str(data_yaml_path),
        split="test",
        imgsz=800,
        plots=True,
        verbose=False,
    )

    prediction_results = model.predict(
        str(test_images_dir),
        save=True,
        conf=0.25,
        project=str(Path(path_to_pretrained_model).parent),
        name="test_predictions",
        exist_ok=True,
        verbose=False,
    )

    print(f"Model: {path_to_pretrained_model}")
    print(f"Test images: {len(image_paths)}")
    print("Metrics:")
    print(f"  Precision:      {float(metrics.box.mp):.4f}")
    print(f"  Recall:         {float(metrics.box.mr):.4f}")
    print(f"  mAP@0.50:       {float(metrics.box.map50):.4f}")
    print(f"  mAP@0.50:0.95:  {float(metrics.box.map):.4f}")
    print(
        f"Saved predictions: {Path(path_to_pretrained_model).parent / 'test_predictions'}"
    )

    sample_count = min(sample_count, len(prediction_results))
    if sample_count == 0:
        print("No predictions were returned by the model.")
        return

    _, axes = plt.subplots(1, sample_count, figsize=(6 * sample_count, 6))
    if sample_count == 1:
        axes = [axes]

    for index in range(sample_count):
        result = prediction_results[index]
        axes[index].imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
        axes[index].set_title(Path(result.path).name)
        axes[index].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
test_annotations_path = str(YOLO_ANNOTATIONS_DIR / "test")
model_path = str(Path(save_model) / "germination_detector.pt")

evaluate_test_images(model_path, test_annotations_path)